In [ ]:
import os

import numpy as np

from vis_utils import read_result_file, INPUT_AGNOSTIC_METHODS

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

START_SEED = 50
NUM_SEEDS = 150

EXPERIMENT_DATASET = 'ycb'
PATH_PREFIX = f'../results/{EXPERIMENT_DATASET}'

# Metrics in paper table order (Tables 3 & 4)
YCB_METRICS = {
    'DCD': '_ycb_dcd',
    'Chamfer L2': '_ycb_cdl2',
    'Chamfer L1': '_ycb_cdl1',
    'IoU': '_ycb_iou',
    'MMD-EMD': '_ycb_emd',
    'Eval3D-geo': '_ycb_geo',
    'Eval3D-struct': '_ycb_struct',
}

METHODS = [
    'Mlp',
    'MF',
    'Knn',
    'Linear',
    'Scout_no_decouple',
    'Scout_proper',
    'Mean',
]

METHOD_SCRIPTS = {
    'Scout_no_decouple': 'scout_no_decouple',
    'Knn': 'knn_script',
    'Mean': 'mean_script',
    'Mlp': 'mlp_script',
    'Linear': 'linear_regression_script',
    'MF': 'mf_script',
    'Scout_proper': 'scout_proper',
}

EXP_NAMES = [
    'single_point_exp',
    'full_cost_range_exp',
    'gen_cost_range_exp',
]


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _collect_exp_values(folder_suffix, method_script, exp_name, seeds):
    """Collect values for a single experiment metric across seeds.

    Args:
        folder_suffix: e.g. '_ycb_dcd'.
        method_script: e.g. 'scout_proper'.
        exp_name: e.g. 'single_point_exp'.
        seeds: Iterable of seed values.

    Returns:
        List of float values across seeds.
    """
    results_dir = f'{PATH_PREFIX}/{method_script}/results{folder_suffix}'
    values = []
    for seed in seeds:
        metrics = read_result_file(os.path.join(results_dir, f'results_{seed}.txt'))
        if metrics is not None and exp_name in metrics:
            values.append(metrics[exp_name])
    return values


def _build_table(exp_names, ycb_metrics, methods, method_scripts, seeds):
    """Build a normalized regret table across metrics and methods.

    Args:
        exp_names: List of experiment names to average over.
        ycb_metrics: Dict of display_name -> folder_suffix.
        methods: List of method display names.
        method_scripts: Dict of display_name -> script folder name.
        seeds: Iterable of seed values.

    Returns:
        Dict of method -> dict of metric_name -> (norm_mean, norm_std_err).
    """
    table = {}

    for metric_display, folder_suffix in ycb_metrics.items():
        baseline_values_per_seed = []
        for exp_name in exp_names:
            vals = _collect_exp_values(folder_suffix, method_scripts['Mean'], exp_name, seeds)
            if not baseline_values_per_seed:
                baseline_values_per_seed = [[v] for v in vals]
            else:
                for i, v in enumerate(vals):
                    baseline_values_per_seed[i].append(v)
        baseline_per_seed = [np.mean(vs) for vs in baseline_values_per_seed]
        baseline_mean = np.mean(baseline_per_seed)

        for method in methods:
            script = method_scripts[method]
            method_values_per_seed = []
            for exp_name in exp_names:
                vals = _collect_exp_values(folder_suffix, script, exp_name, seeds)
                if not method_values_per_seed:
                    method_values_per_seed = [[v] for v in vals]
                else:
                    for i, v in enumerate(vals):
                        method_values_per_seed[i].append(v)
            per_seed = np.array([np.mean(vs) for vs in method_values_per_seed])

            norm_mean = np.mean(per_seed) / baseline_mean
            norm_std_err = (np.std(per_seed) / np.sqrt(len(per_seed))) / baseline_mean

            if method not in table:
                table[method] = {}
            table[method][metric_display] = (norm_mean, norm_std_err)

    return table


def _print_cross_metric_latex(table, ycb_metrics, methods, title):
    """Print a cross-metric LaTeX table.

    Args:
        table: Dict from _build_table.
        ycb_metrics: Dict of metric display names.
        methods: List of method display names.
        title: Title comment for the table.
    """
    metric_names = list(ycb_metrics.keys())
    print(f'% {title}')
    print('% Columns: Method & ' + ' & '.join(metric_names))
    print()

    for method in methods:
        if method not in table:
            continue
        row_parts = [method]
        for metric in metric_names:
            if metric not in table[method]:
                row_parts.append('--')
            elif method in INPUT_AGNOSTIC_METHODS:
                m, _ = table[method][metric]
                row_parts.append(f'${m:.4f}$')
            else:
                m, s = table[method][metric]
                row_parts.append(f'${m:.4f}\\pm{s:.4f}$')
        print(' & '.join(row_parts) + r' \\')
    print()


print('Setup complete.')

In [ ]:
# ---------------------------------------------------------------------------
# Table 3: Regret for C_0 (single_point_exp only)
# Table 4: Regret for full C (average of interesting subspace)
# ---------------------------------------------------------------------------

seeds = range(START_SEED, START_SEED + NUM_SEEDS)
table_c0 = _build_table(EXP_NAMES[:1], YCB_METRICS, METHODS, METHOD_SCRIPTS, seeds)
table_c = _build_table(EXP_NAMES[1:2], YCB_METRICS, METHODS, METHOD_SCRIPTS, seeds)

In [ ]:
# ---------------------------------------------------------------------------
# Pandas summary for quick inspection
# ---------------------------------------------------------------------------

metric_names = list(YCB_METRICS.keys())

header = f"{'Method':<25s} {' '.join(f'{m:>16s}' for m in metric_names)}"

print('Table 3 (C_0):')
print('=' * len(header))
print(header)
print('-' * len(header))
for method in METHODS:
    label = 'Input Agnostic' if method == 'Mean' else method
    if method == 'Mean':
        vals = [f"{table_c0[method][m][0]:.4f}" for m in metric_names]
    else:
        vals = [f"{table_c0[method][m][0]:.4f}±{table_c0[method][m][1]:.4f}" for m in metric_names]
    print(f"{label:<25s} {' '.join(f'{v:>16s}' for v in vals)}")

print()
print('Table 4 (C):')
print('=' * len(header))
print(header)
print('-' * len(header))
for method in METHODS:
    label = 'Input Agnostic' if method == 'Mean' else method
    if method == 'Mean':
        vals = [f"{table_c[method][m][0]:.4f}" for m in metric_names]
    else:
        vals = [f"{table_c[method][m][0]:.4f}±{table_c[method][m][1]:.4f}" for m in metric_names]
    print(f"{label:<25s} {' '.join(f'{v:>16s}' for v in vals)}")